# ENet-4 Jupyter Notebook
Four-variable Elastic Net model with nested CV, OOF predictions, metrics, ORs, VIF, and combined figure.

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import chi2, norm
from math import sqrt
import os


In [ ]:

def delong_roc_variance(ground_truth, predictions):
    order = np.argsort(-predictions)
    predictions = predictions[order]
    ground_truth = ground_truth[order]
    distinct_value_indices = np.where(np.diff(predictions))[0]
    threshold_idxs = np.r_[distinct_value_indices, ground_truth.size - 1]
    tps = np.cumsum(ground_truth)[threshold_idxs]
    fps = 1 + threshold_idxs - tps
    tps = np.r_[0, tps]; fps = np.r_[0, fps]
    P = np.sum(ground_truth); N = ground_truth.size - P
    if P == 0 or N == 0:
        return np.nan, np.nan
    fpr = fps / N; tpr = tps / P
    auc_val = np.trapz(tpr, fpr)
    pos_preds = predictions[ground_truth == 1]
    neg_preds = predictions[ground_truth == 0]
    m = len(pos_preds); n = len(neg_preds)
    def v(u, v_):
        if u > v_: return 1.0
        elif u == v_: return 0.5
        else: return 0.0
    v_vec = np.array([np.mean([v(up, vn) for vn in neg_preds]) for up in pos_preds])
    u_vec = np.array([np.mean([v(up, vn) for up in pos_preds]) for vn in neg_preds])
    auc_cov = (np.var(v_vec, ddof=1)/m) + (np.var(u_vec, ddof=1)/n)
    return auc_val, auc_cov

def delong_ci(y_true, y_score):
    auc_val, auc_var = delong_roc_variance(np.asarray(y_true), np.asarray(y_score))
    if np.isnan(auc_val) or np.isnan(auc_var):
        return np.nan, (np.nan, np.nan)
    se = sqrt(auc_var)
    z = norm.ppf(1 - 0.05/2)
    lower = max(0.0, auc_val - z*se)
    upper = min(1.0, auc_val + z*se)
    return auc_val, (lower, upper)

def brier_score(y, p):
    return float(np.mean((p - y)**2))

def spiegelhalter_test(y, p):
    y = np.asarray(y).astype(float); p = np.asarray(p).astype(float)
    num = np.sum(y - p)
    den = np.sqrt(np.sum(p*(1-p)) + 1e-12)
    Z = num/den if den>0 else np.nan
    pval = 2*(1 - norm.cdf(abs(Z))) if not np.isnan(Z) else np.nan
    return float(Z), float(pval)

def hosmer_lemeshow(y, p, g=10):
    y = np.asarray(y).astype(int); p = np.asarray(p).astype(float)
    df = pd.DataFrame({'y':y,'p':p})
    df['bin'] = pd.qcut(df['p'], q=g, duplicates='drop')
    grp = df.groupby('bin')
    obs = grp['y'].sum().values; n = grp.size().values; exp = grp['p'].sum().values
    chi2_stat = np.sum((obs-exp)**2/(exp+1e-12) + ((n-obs)-(n-exp))**2/((n-exp)+1e-12))
    dfree = len(n) - 2
    pval = 1 - chi2.cdf(chi2_stat, dfree)
    cal_table = pd.DataFrame({'mean_pred': grp['p'].mean().values, 'obs_rate': obs/n})
    return float(chi2_stat), int(dfree), float(pval), cal_table


In [ ]:

data_path = 'Combined demo and imaging data.xlsx'
df = pd.read_excel(data_path, engine='openpyxl')
rename_map = {
    'Lung cancer (0=no, 1=yes)': 'lung_cancer',
    'AIRC nodule malignancy risk (0=low, 1=high)': 'airc_high',
    'Suspicious nodule morphology (0=no, 1=yes)': 'suspicious',
    'Nodule invades adjacent structures on CT (0=no, 1=yes)': 'inv_adj',
    'COPD': 'copd'
}
for col in df.columns:
    if 'emphy' in col.lower() and 'ct' in col.lower():
        rename_map[col] = 'emphysema_ct'

df = df.rename(columns=rename_map)
features4 = ['airc_high','suspicious','inv_adj']
features4.append('emphysema_ct' if 'emphysema_ct' in df.columns else 'copd')
work = df[['lung_cancer']+features4].copy()
for c in work.columns:
    work[c] = pd.to_numeric(work[c], errors='coerce')
for c in features4:
    work[c] = (work[c] > 0).astype(int)
work = work.dropna().copy()
X = work[features4].values
y = work['lung_cancer'].astype(int).values


In [ ]:

outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=321)
inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=654)
Cs = np.logspace(-3,3,20); l1_grid = [0.1,0.3,0.5,0.7,0.9]
oof = np.zeros(len(y))

for tr, te in outer.split(X,y):
    X_tr, X_te = X[tr], X[te]
    y_tr = y[tr]
    best_loss = np.inf; best_pipe=None
    for l1 in l1_grid:
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegressionCV(Cs=Cs, cv=inner, penalty='elasticnet', solver='saga',
                                         l1_ratios=[l1], scoring='neg_log_loss', max_iter=5000, n_jobs=-1))
        ])
        pipe.fit(X_tr, y_tr)
        clf = pipe.named_steps['clf']
        mean_scores = clf.scores_[1].mean(axis=0)
        idx = int(np.argmax(mean_scores))
        cur_loss = -mean_scores[idx]
        if cur_loss < best_loss:
            best_loss = cur_loss; best_pipe = pipe
    oof[te] = best_pipe.predict_proba(X_te)[:,1]

auc,(auc_lo,auc_hi) = delong_ci(y,oof)
bs = brier_score(y,oof)
Z,p_sp = spiegelhalter_test(y,oof)
HL_chi2,HL_df,HL_p,cal_table = hosmer_lemeshow(y,oof,g=10)
auc, auc_lo, auc_hi, bs, Z, p_sp, HL_chi2, HL_df, HL_p


In [ ]:

sm_df = work.copy(); sm_df['intercept']=1.0
res = sm.Logit(sm_df['lung_cancer'], sm_df[['intercept']+features4]).fit(disp=False)
params=res.params; conf=res.conf_int(); pvals=res.pvalues
OR=np.exp(params); CI_low=np.exp(conf[0]); CI_high=np.exp(conf[1])
OR_table=pd.DataFrame({'term':OR.index,'OR':OR.values,'CI_low':CI_low.values,'CI_high':CI_high.values,'p_value':pvals.values})
OR_table = OR_table[OR_table['term']!='intercept']

X_vif = sm_df[features4].astype(float)
VIF = pd.DataFrame({'variable':features4,'VIF':[variance_inflation_factor(X_vif.values,i) for i in range(len(features4))]})

OR_table, VIF


In [ ]:

plt.figure(figsize=(13,4))
fpr,tpr,_ = roc_curve(y,oof)
plt.subplot(1,3,1)
plt.plot(fpr,tpr)
plt.plot([0,1],[0,1],'--')
plt.title('ROC')
plt.subplot(1,3,2)
plt.scatter(cal_table['mean_pred'],cal_table['obs_rate'])
plt.plot([0,1],[0,1],'--')
plt.title('Calibration')
plt.subplot(1,3,3)
plt.errorbar(OR_table['OR'], OR_table.index, xerr=[OR_table['OR']-OR_table['CI_low'], OR_table['CI_high']-OR_table['OR']], fmt='o')
plt.xscale('log')
plt.title('OR Forest')
plt.tight_layout()
plt.show()
